### Question 1: [IPO] Withdrawn IPOs by Company Type

In [2]:
import pandas as pd
import re

# Step 1: Load data from the URL
url = "https://www.iposcoop.com/ipos-recently-filed/"
tables = pd.read_html(url)
df = tables[0]  # Get the first table

# Filter for withdrawn IPOs
df = df[df['Expected To Trade'] == 'Withdrawn'].reset_index(drop=True)

# Step 2: Create Company Type column with categorization rules
def categorize_company_type(company_name):
    rules = [
        ("Technologies", "Technologies"),
        ("Acquisition Corp", "Acquisition Corp"),
        ("Acquisition Corporation", "Acquisition Corp"),
        ("Corp", "Acquisition Corp"),
        ("Inc", "Inc."),
        ("Incorporated", "Inc."),
        ("Group", "Group"),
        ("Ltd", "Limited"),
        ("Limited", "Limited"),
        ("Holdings", "Holdings"),
        ("Holding", "Holdings"),
    ]
    
    for pattern, category in rules:
        if pattern in company_name:
            return category
    return "Other"

company_col = 'Company Name' if 'Company Name' in df.columns else 'Company'
df['Company Type'] = df[company_col].astype(str).apply(categorize_company_type)

# Step 3: Parse price fields and calculate average
def extract_price(price_str):
    if pd.isna(price_str) or price_str == '-':
        return None
    match = re.search(r'\$?([\d.]+)', str(price_str))
    return float(match.group(1)) if match else None

df['Price_Low'] = df['Price Low'].apply(extract_price)
df['Price_High'] = df['Price High'].apply(extract_price)
df['Avg_price'] = df.apply(lambda row: (row['Price_Low'] + row['Price_High']) / 2 
                           if pd.notna(row['Price_Low']) and pd.notna(row['Price_High']) else None, axis=1)

# Step 4: Convert to numeric
def clean_numeric(value):
    if pd.isna(value) or value == '-':
        return None
    value = str(value).replace('$', '').replace(',', '')
    try:
        return float(value)
    except:
        return None

df['Shares_millions'] = df['Shares (millions)'].apply(clean_numeric)
df['Est_vol_millions'] = df['Est $ Vol (millions)'].apply(clean_numeric)

# Step 5: Calculate Shares_offered_value
df['Shares_offered_value'] = df.apply(
    lambda row: row['Shares_millions'] * row['Avg_price'] 
    if pd.notna(row['Shares_millions']) and pd.notna(row['Avg_price']) 
    else row['Est_vol_millions'], 
    axis=1
)

# Step 6: Aggregate by Company Type
result = df.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)

print(result)
print(f"\nHighest: {result.idxmax()} with ${result.max():.2f} million")

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64

Highest: Acquisition Corp with $499.99 million


### Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [3]:
import numpy as np
import yfinance as yf

# 1) Load 2025 IPO pricings
ipo_url = "https://www.iposcoop.com/2025-pricings/"
ipo_2025 = pd.read_html(ipo_url)[0].copy()
ipo_2025.columns = [str(c).strip() for c in ipo_2025.columns]

# Resolve key columns
offer_date_col = "Offer Date" if "Offer Date" in ipo_2025.columns else [c for c in ipo_2025.columns if "Offer" in c and "Date" in c][0]
ticker_col = "Symbol" if "Symbol" in ipo_2025.columns else ("Ticker" if "Ticker" in ipo_2025.columns else [c for c in ipo_2025.columns if "Symbol" in c or "Ticker" in c][0])

# 2) Filter: Offer Date < 2025-09-01 and exclude 0% return rows
ipo_2025[offer_date_col] = pd.to_datetime(ipo_2025[offer_date_col], errors="coerce")
filtered_ipo = ipo_2025[ipo_2025[offer_date_col] < pd.Timestamp("2025-09-01")].copy()

return_candidates = [c for c in filtered_ipo.columns if c.lower() in {"return", "% change", "change", "gain"} or "return" in c.lower()]
if return_candidates:
    ret_col = return_candidates[0]
    ret_num = pd.to_numeric(
        filtered_ipo[ret_col].astype(str).str.replace("%", "", regex=False).str.replace(",", "", regex=False),
        errors="coerce"
    )
    filtered_ipo = filtered_ipo[ret_num.ne(0)]
else:
    print("No return column found; skipping 0% return exclusion.")

# Clean tickers
tickers = (
    filtered_ipo[ticker_col]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"[^A-Z0-9\.\-]", "", regex=True)
)
tickers = tickers[tickers.ne("") & tickers.ne("NAN")].unique().tolist()

print(f"Filtered IPO rows: {len(filtered_ipo)}")
print(f"Unique candidate tickers: {len(tickers)}")

# 3) Download OHLCV from Yahoo (handle missing/delisted)
frames = []
ok_tickers = []

for t in tickers:
    try:
        h = yf.Ticker(t).history(start="2025-01-01", end="2026-09-12", auto_adjust=False)
        if h.empty:
            continue
        h = h.reset_index()
        h["Ticker"] = t
        frames.append(h[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]])
        ok_tickers.append(t)
    except Exception:
        continue

stocks_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
    columns=["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]
)

print(f"Tickers with Yahoo data: {len(ok_tickers)}")

# 4) Feature engineering
stocks_df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
stocks_df["growth_252d"] = stocks_df.groupby("Ticker")["Close"].transform(lambda s: s / s.shift(252))
stocks_df["volatility"] = stocks_df.groupby("Ticker")["Close"].transform(lambda s: s.rolling(30).std() * np.sqrt(252))

# 5) Sharpe ratio (risk-free = 5%)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

# 6) Keep 2026-09-11 and describe
snapshot = stocks_df[stocks_df["Date"].dt.strftime("%Y-%m-%d") == "2026-09-11"].copy()
desc = snapshot[["growth_252d", "volatility", "Sharpe"]].describe()

print(desc)
print("\nMedian Sharpe ratio on 2026-09-11:", snapshot["Sharpe"].median())

Filtered IPO rows: 146
Unique candidate tickers: 146


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found


Tickers with Yahoo data: 132
       growth_252d  volatility      Sharpe
count   130.000000  131.000000  130.000000
mean      1.052117   27.571999         inf
std       3.049217   66.868036         NaN
min       0.001005    0.000000   -0.040147
25%       0.141035    2.301309    0.012042
50%       0.594534    7.885748    0.049016
75%       1.038821   23.968537    0.133619
max      33.638270  627.698106         inf

Median Sharpe ratio on 2026-09-11: 0.049016452355376945


### Question 3: [IPO] 'Fixed Months Holding Strategy'

In [4]:
# Fixed-month holding strategy (1 to 12 months, 21 trading days per month)

trading_days_per_month = 21
months = range(1, 13)

# 1) Future growth features
future_cols = []
for m in months:
    col = f"future_growth_{m}_m"
    stocks_df[col] = stocks_df.groupby("Ticker")["Close"].shift(-m * trading_days_per_month) / stocks_df["Close"]
    future_cols.append(col)

# 2) First trading day (entry point) for each ticker
first_trade = (
    stocks_df.groupby("Ticker", as_index=False)["Date"]
    .min()
    .rename(columns={"Date": "min_date"})
)

# 3) Inner join to keep only IPO entry-day rows with future growth values
entry_rows = stocks_df[["Ticker", "Date", "Close"] + future_cols].copy()
ipo_entry_growth = first_trade.merge(
    entry_rows,
    left_on=["Ticker", "min_date"],
    right_on=["Ticker", "Date"],
    how="inner"
).drop(columns=["Date"])

# 4) Median/mean analysis across holding periods
median_growth = ipo_entry_growth[future_cols].median()
mean_growth = ipo_entry_growth[future_cols].mean()

summary = pd.DataFrame({
    "month": list(months),
    "median_growth": [median_growth[f"future_growth_{m}_m"] for m in months],
    "mean_growth": [mean_growth[f"future_growth_{m}_m"] for m in months],
})

best_idx = summary["median_growth"].idxmax()
best_month = int(summary.loc[best_idx, "month"])
best_median = float(summary.loc[best_idx, "median_growth"])

print("Median vs Mean growth by holding period:")
print(summary.to_string(index=False))

print(f"\nOptimal holding period: {best_month} month(s)")
print(f"Maximum median growth: {best_median:.4f}x")

print("\nQuick trend check:")
print("Median growth sequence:", [round(x, 4) if pd.notna(x) else None for x in summary["median_growth"].tolist()])
print("Mean growth sequence:  ", [round(x, 4) if pd.notna(x) else None for x in summary["mean_growth"].tolist()])

Median vs Mean growth by holding period:
 month  median_growth  mean_growth
     1       0.935351    95.751507
     2       0.892958    67.978625
     3       0.827160    51.247805
     4       0.730400    22.697288
     5       0.690909    54.242665
     6       0.700916    64.010844
     7       0.659973    59.265533
     8       0.603508    21.457209
     9       0.580294    20.006901
    10       0.533333    11.256621
    11       0.481811     8.899609
    12       0.491838     5.818939

Optimal holding period: 1 month(s)
Maximum median growth: 0.9354x

Quick trend check:
Median growth sequence: [0.9354, 0.893, 0.8272, 0.7304, 0.6909, 0.7009, 0.66, 0.6035, 0.5803, 0.5333, 0.4818, 0.4918]
Mean growth sequence:   [95.7515, 67.9786, 51.2478, 22.6973, 54.2427, 64.0108, 59.2655, 21.4572, 20.0069, 11.2566, 8.8996, 5.8189]


### Question 4: [Strategy] Simple RSI-Based Trading Strategy

In [6]:
! pip install -q gdown

In [7]:
import gdown


# 1) Download and load data
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
strategy_df = pd.read_parquet("data.parquet", engine="pyarrow")

# 2) Resolve key columns robustly
date_col = "Date" if "Date" in strategy_df.columns else "date"
rsi_col = "rsi" if "rsi" in strategy_df.columns else [c for c in strategy_df.columns if "rsi" in c.lower()][0]
growth_col = "growth_future_30d"

# 3) Filter period + oversold signal
strategy_df[date_col] = pd.to_datetime(strategy_df[date_col], errors="coerce")
selected_df = strategy_df[
    (strategy_df[date_col] >= "2000-01-01") &
    (strategy_df[date_col] <= "2025-06-01") &
    (strategy_df[rsi_col] < 30) &
    (strategy_df[growth_col].notna())
].copy()

# 4) Profit calculation
net_income = 1000 * (selected_df[growth_col] - 1).sum()
net_income_k = net_income / 1000

# Optional diagnostics
avg_return = (selected_df[growth_col] - 1).mean()
win_rate = (selected_df[growth_col] > 1).mean()

print(f"Trades: {len(selected_df)}")
print(f"Average 30d return: {avg_return:.4%}")
print(f"Win rate: {win_rate:.2%}")
print(f"Net income: ${net_income:,.2f}")
print(f"Net income (in $ thousands): {net_income_k:,.2f}")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=6f06348b-b955-41e2-a046-ac9678db9f00
To: /Users/yoni/Documents/Stock-market-zoomcamp/data.parquet
100%|██████████| 130M/130M [00:04<00:00, 26.4MB/s] 


Trades: 5206
Average 30d return: 1.2640%
Win rate: 55.13%
Net income: $65,805.59
Net income (in $ thousands): 65.81


### Q5. [Exploratory, Optional] Predicting a Positive-Return IPO

#### Strategies to Increase IPO Investing Profitability

 - Implement Quality and Fundamental Filters (Screening)



 - Short-Selling or Hedging (Betting Against Hype)



 - Post-Lockup Expiration Entry



 - Utilize Technical Momentum Indicators



 - The 'Underpricing Flip' (Primary Market Allocation)

